# 📌 PySpark Window Functions Cheat Sheet — `lag()`, `lead()` & Running Aggregations

| Function | What it does | Example |
|----------|--------------|---------|
| `lag(col, n)` | Returns the value from **n previous rows** | Previous order amount |
| `lead(col, n)` | Returns the value from **n next rows** | Next order amount |
| `sum(col).over(window)` | Running (cumulative) total | Running revenue |
| `avg(col).over(window)` | Running average | Running average sales |
| `count(col).over(window)` | Running count | Orders processed so far |
| `min(col).over(window)` | Running minimum | Lowest price so far |
| `max(col).over(window)` | Running maximum | Highest price so far |
| `first(col)` | First value in the window | First order amount |
| `last(col)` | Last value in the window | Latest order amount |

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-18")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a3d79b41-5397-441c-b25a-6fa75a5f4146;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 147ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

For each customer, use lag() to get the previous order's unit_price. Calculate the difference between the current and previous price. Show customers where the price increased (positive difference).

In [2]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.partitionBy("customer_id").orderBy(F.col("order_date").asc())

df = orders_df.withColumn(
    "previous_unit_price",
    F.lag("unit_price", 1).over(window_spec)
)

df = df.withColumn(
    "Increased_price",
    F.col("unit_price") - F.col("previous_unit_price")
)

df.filter(F.col("Increased_price") > 0) \
  .select(
      "customer_id",
      "order_id",
      "order_date",
      "unit_price",
      "previous_unit_price",
      "Increased_price"
  ).show()

+-----------+--------+----------+----------+-------------------+------------------+
|customer_id|order_id|order_date|unit_price|previous_unit_price|   Increased_price|
+-----------+--------+----------+----------+-------------------+------------------+
|       C001|   O0046|2023-05-17|    109.99|              89.99|              20.0|
|       C001|   O0071|2023-08-01|    449.99|             109.99|             340.0|
|       C002|   O0047|2023-05-20|     79.99|              59.99|19.999999999999993|
|       C002|   O0072|2023-08-04|   1299.99|              79.99|            1220.0|
|       C003|   O0073|2023-08-07|     49.99|              29.99|20.000000000000004|
|       C003|   O0098|2023-10-22|    699.99|              49.99|             650.0|
|       C004|   O0024|2023-03-12|   1299.99|              89.99|            1210.0|
|       C005|   O0025|2023-03-15|    109.99|              29.99|              80.0|
|       C005|   O0075|2023-08-13|    599.99|              59.99|            

**Task 2**

For each customer, use lead() to show what the next order's unit_price will be. For the last order of each customer, the next price should default to 0.0 instead of null.

In [3]:
orders_df.select(
    "customer_id",
    "order_id",
    "order_date",
    F.col('unit_price').alias('current_price'),
    F.lead("unit_price",1,0).over(Window.partitionBy('customer_id').orderBy('order_date')).alias("next_order_price")
).show(truncate=False)

+-----------+--------+----------+-------------+----------------+
|customer_id|order_id|order_date|current_price|next_order_price|
+-----------+--------+----------+-------------+----------------+
|C001       |O0001   |2023-01-05|1299.99      |89.99           |
|C001       |O0021   |2023-03-03|89.99        |109.99          |
|C001       |O0046   |2023-05-17|109.99       |449.99          |
|C001       |O0071   |2023-08-01|449.99       |349.99          |
|C001       |O0096   |2023-10-16|349.99       |0.0             |
|C002       |O0002   |2023-01-07|449.99       |59.99           |
|C002       |O0022   |2023-03-06|59.99        |79.99           |
|C002       |O0047   |2023-05-20|79.99        |1299.99         |
|C002       |O0072   |2023-08-04|1299.99      |89.99           |
|C002       |O0097   |2023-10-19|89.99        |0.0             |
|C003       |O0003   |2023-01-10|349.99       |79.99           |
|C003       |O0023   |2023-03-09|79.99        |29.99           |
|C003       |O0048   |202

**Task 3**

Calculate a running total of unit_price per region ordered by order_date. Show region, order_date, unit_price, and cumulative_revenue.

In [4]:
window_spec = Window \
    .partitionBy("region") \
    .orderBy(F.col("order_date").asc()) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

orders_df.withColumn(
    "revenue",
    F.col("unit_price") * F.col("quantity")
).select(
    "region",
    "order_date",
    "unit_price",
    "revenue",
    F.sum("revenue")
     .over(window_spec)
     .alias("cumulative_revenue")
).show(truncate=False)

+------+----------+----------+------------------+------------------+
|region|order_date|unit_price|revenue           |cumulative_revenue|
+------+----------+----------+------------------+------------------+
|East  |2023-01-05|1299.99   |2599.98           |2599.98           |
|East  |2023-01-18|199.99    |199.99            |2799.9700000000003|
|East  |2023-02-05|599.99    |599.99            |3399.96           |
|East  |2023-02-14|699.99    |699.99            |4099.95           |
|East  |2023-02-28|449.99    |899.98            |4999.93           |
|East  |2023-03-03|89.99     |89.99             |5089.92           |
|East  |2023-03-18|349.99    |699.98            |5789.9            |
|East  |2023-04-05|29.99     |179.94            |5969.839999999999 |
|East  |2023-04-14|89.99     |269.96999999999997|6239.8099999999995|
|East  |2023-04-29|59.99     |179.97            |6419.78           |
|East  |2023-05-08|199.99    |199.99            |6619.7699999999995|
|East  |2023-05-14|89.99     |359.

**Task 4**

For each customer, calculate the running count of orders and running average spend. Show the first 3 orders per customer — which customer reached the highest running average fastest?

In [5]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

df = orders_df.withColumn(
    "spend",
    F.col("unit_price") * F.col("quantity")*(1-F.col('discount_pct')/100)
).withColumn(
    "order_count",
    F.count("order_id").over(
        window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
).withColumn(
    "average_spend",
    F.avg("spend").over(
        window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
).withColumn(
    "rn",
    F.row_number().over(window_spec)
)

df.filter(F.col("rn") <= 3).show()

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+------------------+-----------+------------------+---+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|             spend|order_count|     average_spend| rn|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+------------------+-----------+------------------+---+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|          2339.982|          1|          2339.982|  1|
|   O0021|       C001|      P006|2023-03-03|       1|     89.99|           0|Delivered|   Credit Card|   East|             89.99|          2|1214.9859999999999|  2|
|   O0046|       C001|      P010|2023-05-17|       2|    109.99|           0|Delivered|   Credit Card|   East|            219.98|          3| 883.3173333333333|  3|
|   O0002|

In [6]:
spark.stop()